# Contrastive Training with Groove Radar (Simplified)

Train the StepMania difficulty classifier with contrastive learning using groove radar similarity.

**Multi-task learning:**
- Classification loss on anchor samples (CrossEntropy)
- Contrastive loss on triplets (TripletMarginLoss with adaptive margins)

**Key components:**
- `GrooveRadar`: DDR-style 5-value feature vector (Stream, Voltage, Air, Freeze, Chaos)
- `TripletSelector`: Selects positive/negative pairs based on groove radar similarity
- `ContrastiveTripletDataset`: Serves triplets for contrastive learning

In [1]:
# Standard imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

# Project imports - NO PATH HACKS NEEDED!
from src.notebook_utils import setup_contrastive_experiment, setup_data_splits
from src.visualization.plots import plot_training_curves, plot_triplet_margins, plot_embedding_space
from src.data import DIFFICULTY_NAMES

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Quick Experiment Setup

Choose experiment A or B and set up everything in one line!

In [2]:
# ===== EXPERIMENT SELECTION =====
# - 'A': Aggressive (freeze classifier head during warmup, downweight classification)
# - 'B': Conservative (selective unfreezing, freeze encoders only)
EXPERIMENT = 'B'  # Change to 'A' for experiment A

# One-line experiment setup - replaces ~300 lines of boilerplate!
exp = setup_contrastive_experiment(
    experiment_config=f'contrastive_experiment_{EXPERIMENT.lower()}',
    pretrained_checkpoint='classifier_baseline',  # Load pretrained classifier
    device='auto',  # Auto-detect CUDA
    batch_size=None,  # Use config default
    num_workers=4,
    verbose=True
)

# Unpack for convenience
model = exp['model']
trainer = exp['trainer']
train_loader = exp['train_loader']
val_loader = exp['val_loader']
config = exp['config']
device = exp['device']

print(f"\n{'='*60}")
print(f"Experiment {EXPERIMENT} ready to train!")
print(f"{'='*60}")

Loaded config: /home/ybx/code/stepmania-chart-generator/config/contrastive_experiment_b.yaml

Setting up datasets...
Loading data from: /home/ybx/code/stepmania-chart-generator/data
Cache directory: /home/ybx/code/stepmania-chart-generator/cache/samples
Found 6360 chart files

Chart file splits:
  Train: 4452
  Val: 954
  Test: 954
Parsing 4452 chart files...
squartatrice failed song length requirement
Howling the Nightmare (bms edit) failed song length requirement
Cube of mind failed song length requirement
ボッカデラベリタ failed song length requirement
Cross Galaxy failed song length requirement
Memoria(b-UMB HARDCORE MIX) failed bpm requirement (avg_bpm=205.0)
Missing Girl failed song length requirement
Cosmic Magic Shooter failed song length requirement
輝夜姫 failed song length requirement
Myself failed song length requirement
ボスとのレース failed song length requirement
KAERU failed bpm requirement (avg_bpm=229.2)
L.E.F. (Loud Electronic Ferocious) failed valid chart requirement (no dance-single

[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?


Ideal forever failed song length requirement
Bass Slut (Original Mix) failed song length requirement
Error processing /home/ybx/code/stepmania-chart-generator/data/community/Mizuki's Simfiles/HAPPY｡ﾙLONG｡ﾙNOTES/ok.sm: Audio file not found: /home/ybx/code/stepmania-chart-generator/data/community/Mizuki's Simfiles/HAPPY｡ﾙLONG｡ﾙNOTES/HAPPYLUCKYBABY.mp3
Sayonara(ALL PHASE MIX) failed song length requirement
Error processing /home/ybx/code/stepmania-chart-generator/data/community/sabo/Jealousy of Green Eyes/Jealousy of Green Eyes.sm: Audio file not found: /home/ybx/code/stepmania-chart-generator/data/community/sabo/Jealousy of Green Eyes/05 Ί̃WFV[.mp3
☆虹色ロックンロール♪ Ex failed valid chart requirement (no dance-single charts)
next to you failed song length requirement
Error processing /home/ybx/code/stepmania-chart-generator/data/community/Gochamaze Greatest Gigapack 2/(ShokuN)theater D/theater D.sm: Audio file not found: /home/ybx/code/stepmania-chart-generator/data/community/Gochamaze Greatest

[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?


ギー太に首ったけ failed song length requirement
繧ｨ繧｢繝ｼ繝槭Φ縺悟偵○縺ｪ縺繧ｫ繝ｩ繧ｪ繧ｱ繝舌繧ｸ繝ｧ繝ｳ failed bpm requirement (avg_bpm=200.0)
Error processing /home/ybx/code/stepmania-chart-generator/data/community/sabo/25236/25236.sm: Audio file not found: /home/ybx/code/stepmania-chart-generator/data/community/sabo/25236/04 tOiCg.mp3
RESTRAINT failed song length requirement
sacrifice Love failed bpm requirement (avg_bpm=253.4)
縺ｪ縺ｪ縺ｍ縺ｳ繧医ｊ(YL-Beatz HandzUp Remix)Short Edit failed song length requirement
レンジで好吃☆電子調理器使用中華料理四千年歴史瞬間調理完了武闘的料理長☆ failed song length requirement
バンブーソード・ガール failed bpm requirement (avg_bpm=208.0)
Error processing /home/ybx/code/stepmania-chart-generator/data/community/remi/âëâuâëâuüÖêñÅεÉΘî╛üI/âëâuâëâuüÖêñÅεÉΘî╛.sm: Audio file not found: /home/ybx/code/stepmania-chart-generator/data/community/remi/âëâuâëâuüÖêñÅεÉΘî╛üI/uu錾.ogg
ベースラインやってる？笑 failed song length requirement
Error processing /home/ybx/code/stepmania-chart-generator/data/community/sabo/1111 (7)/111.sm: Audio file not found: /home/

...

Caching samples: 100%|████████████████████████| 786/786 [04:31<00:00,  2.90it/s]


Cache warming complete: 786 new, 0 existing
Warming cache to /home/ybx/code/stepmania-chart-generator/cache/samples/test...


Caching samples: 100%|████████████████████████| 770/770 [04:32<00:00,  2.82it/s]


Cache warming complete: 770 new, 0 existing
Cache warming complete!

Creating contrastive triplet dataset...
Fitting triplet selector...
TripletSelector fitted: positive_threshold=0.0779, negative_threshold=0.3105
Triplet selection stats: 3803/3820 (99.6%) samples can form valid triplets
Precomputing triplets...
Precomputed 3803 triplets

DataLoaders created: train=29 batches, val=7 batches

Creating model...

Loading pretrained weights from: classifier_baseline


FileNotFoundError: [Errno 2] No such file or directory: 'classifier_baseline'

## 2. Optional: Inspect Configuration

In [ ]:
# Print key configuration settings
print("\n=== Training Config (Key Settings) ===")
training_config = config['training']
for k in ['batch_size', 'learning_rate', 'num_epochs', 'warmup_epochs',
          'warmup_cls_weight', 'finetune_cls_weight', 'selective_unfreeze']:
    if k in training_config:
        print(f"  {k}: {training_config[k]}")

print("\n=== Contrastive Config ===")
contrastive_config = config['contrastive']
for k in ['classification_weight', 'contrastive_weight', 'triplet_margin',
          'positive_percentile', 'negative_percentile']:
    if k in contrastive_config:
        print(f"  {k}: {contrastive_config[k]}")

## 3. Train!

Everything is set up and ready to go.

In [ ]:
# Train the model
history = trainer.fit(epochs=config['training']['num_epochs'])

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"  Best validation loss: {min(history['val_loss']):.4f}")
print(f"  Best validation accuracy: {max(history['val_acc']):.4f}")

## 4. Training Analysis

Visualize training progress with reusable plotting functions.

In [ ]:
# Plot training curves using visualization utility
plot_training_curves(
    history,
    metrics=['loss', 'acc', 'cls_loss', 'contrastive_loss'],
    title=f'Experiment {EXPERIMENT} Training History'
)
plt.show()

## 5. Embedding Space Visualization

Visualize the learned embedding space using t-SNE.

In [ ]:
# Extract embeddings from validation set
@torch.no_grad()
def extract_embeddings(model, dataloader, device, max_samples=500):
    """Extract embeddings and labels from a dataloader."""
    model.eval()
    
    all_embeddings = []
    all_labels = []
    
    total = 0
    for batch in tqdm(dataloader, desc="Extracting embeddings"):
        audio = batch['audio'].to(device)
        chart = batch['chart'].to(device)
        mask = batch['mask'].to(device)
        labels = batch['difficulty']
        groove_radar = batch.get('groove_radar')
        if groove_radar is not None:
            groove_radar = groove_radar.to(device)
        
        output = model(audio, chart, mask, groove_radar=groove_radar, return_embeddings=True)
        
        all_embeddings.append(output['embeddings'].cpu().numpy())
        all_labels.append(labels.numpy())
        
        total += labels.size(0)
        if total >= max_samples:
            break
    
    embeddings = np.concatenate(all_embeddings, axis=0)[:max_samples]
    labels = np.concatenate(all_labels, axis=0)[:max_samples]
    
    return embeddings, labels

embeddings, labels = extract_embeddings(model, val_loader, device)
print(f"Extracted {len(embeddings)} embeddings")

In [ ]:
# Plot embedding space using visualization utility
plot_embedding_space(
    embeddings,
    labels,
    method='tsne',
    color_by='difficulty',
    labels_names=DIFFICULTY_NAMES,
    title=f'Embedding Space - Experiment {EXPERIMENT}'
)
plt.show()

## 6. Triplet Quality Analysis

Analyze how well the contrastive learning separates samples.

In [ ]:
# Compute embedding distances for triplets
@torch.no_grad()
def analyze_triplet_distances(model, train_loader, device, n_batches=10):
    """Analyze distances between anchor-positive and anchor-negative in embedding space."""
    model.eval()
    
    ap_distances = []  # Anchor-Positive distances
    an_distances = []  # Anchor-Negative distances
    
    for i, batch in enumerate(train_loader):
        if i >= n_batches:
            break
            
        # Get embeddings for anchor, positive, negative
        anchor_out = model(
            batch['anchor_audio'].to(device),
            batch['anchor_chart'].to(device),
            batch['anchor_mask'].to(device),
            groove_radar=batch['anchor_groove_radar'].to(device),
            return_embeddings=True
        )
        positive_out = model(
            batch['positive_audio'].to(device),
            batch['positive_chart'].to(device),
            batch['positive_mask'].to(device),
            groove_radar=batch['positive_groove_radar'].to(device),
            return_embeddings=True
        )
        negative_out = model(
            batch['negative_audio'].to(device),
            batch['negative_chart'].to(device),
            batch['negative_mask'].to(device),
            groove_radar=batch['negative_groove_radar'].to(device),
            return_embeddings=True
        )
        
        # Compute distances
        ap_dist = torch.norm(anchor_out['embeddings'] - positive_out['embeddings'], dim=1)
        an_dist = torch.norm(anchor_out['embeddings'] - negative_out['embeddings'], dim=1)
        
        ap_distances.extend(ap_dist.cpu().numpy().tolist())
        an_distances.extend(an_dist.cpu().numpy().tolist())
    
    return np.array(ap_distances), np.array(an_distances)

ap_distances, an_distances = analyze_triplet_distances(model, train_loader, device)
print(f"Analyzed {len(ap_distances)} triplets")

In [ ]:
# Plot triplet margins using visualization utility
target_margin = config['contrastive']['triplet_margin']
plot_triplet_margins(ap_distances, an_distances, target_margin=target_margin)
plt.show()

# Print statistics
margins = an_distances - ap_distances
satisfied = (margins > 0).sum()
print(f"\nTriplet Statistics:")
print(f"  Mean AP distance: {ap_distances.mean():.4f}")
print(f"  Mean AN distance: {an_distances.mean():.4f}")
print(f"  Mean margin: {margins.mean():.4f}")
print(f"  Triplets with positive margin: {satisfied}/{len(margins)} ({satisfied/len(margins):.1%})")

## Summary

This simplified notebook replaces **~300 lines of setup boilerplate** with a **single function call** (`setup_contrastive_experiment()`), making experiments easier to run and modify.

**Key improvements:**
- No path hacks needed (`sys.path.insert`)
- No manual dataset creation, cache warming, or dataloader setup
- No manual model creation or pretrained weight loading
- No manual trainer initialization
- Reusable plotting functions for consistent visualizations

**To explore in detail:**
- See `contrastive_training.ipynb` for detailed groove radar analysis
- See `src/notebook_utils.py` for implementation details
- See `src/visualization/plots.py` for all plotting functions